In [2]:
%matplotlib inline
import os
import sys
from os.path import join as pjoin
from tifffile import imread, imwrite, TiffFile
import numpy as np
import shutil
import matplotlib.pyplot as plt
from glob import glob
import pandas as pd
import cv2
from tqdm import tqdm
import subprocess
from scipy.ndimage import gaussian_filter,median_filter
from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt

project_root = '/home/lsh/WF_GoNogo'
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.wfield_utils import *

In [3]:
# batch_widefield_preproc_with_outlier.py
import os
from os.path import join as pjoin
from glob import glob
import numpy as np
from tifffile import imread, imwrite
import matplotlib.pyplot as plt
import yaml
import time
import re

# ------------------------------
# Funtions
# ------------------------------
def load_config(config_path):
    """load the YAML config"""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

def filename2int(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

# ------------------------------
# Step 1: organize tiffes
# ------------------------------
# def organize_tif(folder_path, processPath):
#     folder_name = os.path.basename(folder_path)
#     tif_path = folder_path + '.tif'
#     if os.path.exists(tif_path):
#         print(f'importing {tif_path}')
#         image_stack = imread(tif_path)
#         print(f'finish importing {tif_path}')
#     else:
#         image_path_ls = glob(os.path.join(folder_path, '*.tif'))
#         image_path_ls = sorted(image_path_ls, key=filename2int)
#         image_stack = [imread(tiff) for tiff in image_path_ls]
#     mean_values = [np.mean(frame) for frame in image_stack]
#     output_value = os.path.join(processPath, folder_name + "-Values.csv")
#     np.savetxt(output_value, mean_values, delimiter=",")
#     return np.array(image_stack)



def merge_two_channels(rawPath, processPath, timePath, experiment):
    """Merge two channels (405 & 470) after dropping unmatched frames."""
    os.makedirs(os.path.join(processPath, experiment + "-wfield"), exist_ok=True)
    mergePath = os.path.join(processPath, experiment + "-wfield")
    merge_file = os.path.join(mergePath, experiment + "-merged.tif")

    if os.path.exists(merge_file):
        print(f"Merged {experiment} already exists")
        return merge_file

    folder_405 = os.path.join(rawPath, experiment + "-405")
    folder_470 = os.path.join(rawPath, experiment + "-470")

    paths_405 = sorted(glob(os.path.join(folder_405, "*.tif")), key=filename2int)
    paths_470 = sorted(glob(os.path.join(folder_470, "*.tif")), key=filename2int)
    idx_405 = np.array([filename2int(p) for p in paths_405])
    idx_470 = np.array([filename2int(p) for p in paths_470])

    timestamps_470 = np.load(pjoin(timePath, "widefield_timestamps_blue.npy"))
    timestamps_405 = np.load(pjoin(timePath, "widefield_timestamps_violet.npy"))

    common_idx = np.intersect1d(idx_405, idx_470)

    print(f"common_idx：{len(common_idx)} (405={len(idx_405)}, 470={len(idx_470)})")
    missing_405 = set(idx_470) - set(idx_405)
    missing_470 = set(idx_405) - set(idx_470)
    print(f"405 missing {len(missing_405)} frames: {sorted(list(missing_405))[:10]}")
    print(f"470 missing {len(missing_470)} frames: {sorted(list(missing_470))[:10]}")

    if len(common_idx) == 0:
        raise ValueError("No overlapping frames between 405 and 470 channels!")

    valid_405 = [os.path.join(folder_405, f"{i}.tif") for i in common_idx if os.path.exists(os.path.join(folder_405, f"{i}.tif"))]
    valid_470 = [os.path.join(folder_470, f"{i}.tif") for i in common_idx if os.path.exists(os.path.join(folder_470, f"{i}.tif"))]
    
    idx2time_470 = dict(zip(idx_470, timestamps_470))
    idx2time_405 = dict(zip(idx_405, timestamps_405))
    time_470 = [idx2time_470[i] for i in common_idx if i in idx2time_470]
    time_405 = [idx2time_405[i] for i in common_idx if i in idx2time_405]


    tif_405 = np.array([imread(p) for p in valid_405])
    tif_470 = np.array([imread(p) for p in valid_470])


    merged_tif = np.stack([tif_470, tif_405], axis=1)  # shape = (frames, 2, H, W)
    imwrite(merge_file, merged_tif, imagej=True, bigtiff=True)

    # Save mean values for outlier detection
    mean_470 = [np.mean(frame) for frame in tif_470]
    mean_405 = [np.mean(frame) for frame in tif_405]
    np.savetxt(os.path.join(processPath, f"{experiment}-470-Values.csv"), mean_470, delimiter=",")
    np.savetxt(os.path.join(processPath, f"{experiment}-405-Values.csv"), mean_405, delimiter=",")
    np.save(os.path.join(timePath, "widefield_timestamps_blue.npy"), np.array(time_470))
    np.save(os.path.join(timePath, "widefield_timestamps_violet.npy"),np.array(time_405))

    print(f"Merge completed: {merge_file}")
    print(f"Final frames: {merged_tif.shape[0]}")

    return merge_file


# ------------------------------
# Step 2: Outlier dection and correction
# ------------------------------
def detect_outlier(mean_values, std_thr=5, qc_path=None, plot=True):
    def find_continuous_outliers(outlier_idx):
        if len(outlier_idx) == 0:
            return []
        segments, start = [], outlier_idx[0]
        for i in range(1, len(outlier_idx)):
            if outlier_idx[i] != outlier_idx[i-1] + 1:
                segments.append((start, outlier_idx[i-1]))
                start = outlier_idx[i]
        segments.append((start, outlier_idx[-1]))
        return segments

    outlier_470 = mean_values[:,0] < mean_values[:,0].mean() - std_thr*mean_values[:,0].std()
    outlier_405 = (mean_values[:,1] < mean_values[:,1].mean() - std_thr*mean_values[:,1].std()) | \
                  (mean_values[:,1] > mean_values[:,1].mean() + std_thr*mean_values[:,1].std())
    outlier_470_segments = find_continuous_outliers(np.where(outlier_470)[0])
    outlier_405_segments = find_continuous_outliers(np.where(outlier_405)[0])

    if plot:
        fig, ax = plt.subplots(figsize=(20,5))
        ax.plot(mean_values[:,0], label='470', color='g')
        ax.plot(mean_values[:,1], label='405', color='purple')
        for s,e in outlier_470_segments:
            ax.plot(np.arange(s,e+1), mean_values[s:e+1,0], 'ko', fillstyle='none')
        for s,e in outlier_405_segments:
            ax.plot(np.arange(s,e+1), mean_values[s:e+1,1], 'ro', fillstyle='none')
        ax.legend()
        plt.title('Raw Mean Values with Outliers')
        
    if qc_path is not None:
            os.makedirs(qc_path, exist_ok=True)
            save_path = os.path.join(qc_path, f"outlier_detection.png")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig) 
            print(f"Outlier dection plot saved to: {save_path}")

    return outlier_470_segments, outlier_405_segments

def correct_lum_outlier(merged_tif, outlier_index_470, outlier_index_405, qc_path, plot=True):
    """Detect and correct luminance outliers, then save corrected data as .bin."""
    data = imread(merged_tif)  # (nframes, nchannels, H, W)
    n_frames, n_channels, H, W = data.shape
    print("Merged frames:", n_frames)

    # --- Outlier correction ---
    for start, end in outlier_index_470:
        data[start:end+1, 0, :, :] = 0.5 * data[start-1, 0, :, :] + 0.5 * data[end+1, 0, :, :]
    for start, end in outlier_index_405:
        data[start:end+1, 1, :, :] = 0.5 * data[start-1, 1, :, :] + 0.5 * data[end+1, 1, :, :]

    # --- Convert to uint16 if not already ---
    if data.dtype != np.uint16:
        data = np.clip(data, 0, np.iinfo(np.uint16).max).astype(np.uint16)

    # --- Define .bin filename ---
    corrected_file = os.path.join(
        os.path.dirname(merged_tif),
        f"{n_frames}_{n_channels}_{H}_{W}_uint16.bin"
    )

    # --- Save as binary file ---
    data.tofile(corrected_file)
    print(f"Corrected .bin file saved: {corrected_file}")

    # --- Optional: Plot and save QC ---
    if plot:
        mean_values_corrected = np.stack([
            data[:, 0].mean(axis=(1, 2)),
            data[:, 1].mean(axis=(1, 2))
        ], axis=1)
        fig, ax = plt.subplots(figsize=(20, 5))
        ax.plot(mean_values_corrected[:, 0], label='470', color='g')
        ax.plot(mean_values_corrected[:, 1], label='405', color='purple')
        ax.legend()
        plt.title('Corrected Mean Values')

        if qc_path is not None:
            os.makedirs(qc_path, exist_ok=True)
            save_path = os.path.join(qc_path, "outlier_correction.png")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"Outlier correction plot saved to: {save_path}")

    return corrected_file


# ------------------------------
# Step 3: Downsample and run SVD
# ------------------------------
from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt
import os

def mmap_dat(filename,
             mode = 'r',
             nframes = None,
             shape = None,
             dtype='uint16'):
    '''
    Loads frames from a binary file as a memory map.
    This is useful when the data does not fit to memory.
    
    Inputs:
        filename (str)       : fileformat convention, file ends in _NCHANNELS_H_W_DTYPE.dat
        mode (str)           : memory map access mode (default 'r')
                'r'   | Open existing file for reading only.
                'r+'  | Open existing file for reading and writing.                 
        nframes (int)        : number of frames to read (default is None: the entire file)
        offset (int)         : offset frame number (default 0)
        shape (list|tuple)   : dimensions (NCHANNELS, HEIGHT, WIDTH) default is None
        dtype (str)          : datatype (default uint16) 
    Returns:
        A memory mapped  array with size (NFRAMES,NCHANNELS, HEIGHT, WIDTH).

    Example:
        dat = mmap_dat(filename)
    '''
    
    if not os.path.isfile(filename):
        raise OSError('File {0} not found.'.format(filename))
    if shape is None or dtype is None: # try to get it from the filename
        dtype,shape,_ = _parse_binary_fname(filename,
                                            shape = shape,
                                            dtype = dtype)
    if type(dtype) is str:
        dt = np.dtype(dtype)
    else:
        dt = dtype
    if nframes is None:
        # Get the number of samples from the file size
        nframes = int(os.path.getsize(filename)/(np.prod(shape)*dt.itemsize))
    dt = np.dtype(dtype)
    return np.memmap(filename,
                     mode=mode,
                     dtype=dt,
                     shape = (int(nframes),*shape))

def _parse_binary_fname(fname,lastidx=None, dtype = 'uint16', shape = None, sep = '_'):
    '''
    Gets the data type and the shape from the filename 
    This is a helper function to use in load_dat.
    
    out = _parse_binary_fname(fname)
    
    With out default to: 
        out = dict(dtype=dtype, shape = shape, fnum = None)
    '''
    fn = os.path.splitext(os.path.basename(fname))[0]
    fnsplit = fn.split(sep)
    fnum = None
    if lastidx is None:
        # find the datatype first (that is the first dtype string from last)
        lastidx = -1
        idx = np.where([not f.isnumeric() for f in fnsplit])[0]
        for i in idx[::-1]:
            try:
                dtype = np.dtype(fnsplit[i])
                lastidx = i
            except TypeError:
                pass
    if dtype is None:
        dtype = np.dtype(fnsplit[lastidx])
    # further split in those before and after lastidx
    before = [f for f in fnsplit[:lastidx] if f.isdigit()]
    after = [f for f in fnsplit[lastidx:] if f.isdigit()]
    if shape is None:
        # then the shape are the last 3
        shape = [int(t) for t in before[-3:]]
    if len(after)>0:
        fnum = [int(t) for t in after]
    return dtype,shape,fnum

def svd_widefield(processPath, experiment, corrected_file,n_colors=None,downsample_factor=15, skip_frames=100, max_components=2000):
    """
    Compute SVD for widefield data stored as (nframes, nchannels, H, W)
    
    dat: np.array or memmap of shape (nframes, nchannels, H, W)
    n_colors: number of colors / channels, if None, use dat.shape[1]
    downsample_factor: frames to skip when computing U
    skip_frames: skip frames at start and end to avoid artifacts
    max_components: max number of SVD components to keep

    Returns:
        U: list of np.array, each (H, W, n_components)
        V: list of np.array, each (n_components, nframes) per channel
        im_avg: list of np.array, average image per channel (H, W)
    """
    mergePath = os.path.join(processPath, experiment + "-wfield")
    if not os.path.isdir(mergePath):
        raise FileNotFoundError(f"{mergePath} not found")

    dat = mmap_dat(corrected_file)
    nframes, nchannels, H, W = dat.shape
    if n_colors is None:
        n_colors = nchannels
    
    U_list = []
    V_list = []
    im_avg_list = []
    
    for ch in range(n_colors):
        print(f"Processing SVD channel {ch+1}/{n_colors}...")
        # select frames for downsampled U calculation
        frame_idx = np.arange(skip_frames, nframes - skip_frames, downsample_factor)
        X = dat[frame_idx, ch, :, :].reshape(len(frame_idx), -1).T  # (H*W, nframes_ds)
        
        # compute mean image for channel
        im_avg = np.mean(X, axis=1).reshape(H, W)
        im_avg_list.append(im_avg)
        
        # subtract mean
        X_centered = X - X.mean(axis=1, keepdims=True)
        
        # SVD
        U_full, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        n_comp = min(max_components, U_full.shape[1])
        U = U_full[:, :n_comp].reshape(H, W, n_comp)
        U_list.append(U)
        
        # project full data to U to get V
        dat_ch = dat[:, ch, :, :].reshape(nframes, -1).T  # (H*W, nframes)
        dat_ch_centered = dat_ch - im_avg.reshape(-1,1)
        V = U.reshape(H*W, n_comp).T @ dat_ch_centered  # (n_comp, nframes)
        V_list.append(V)
        if ch == 0:
            suffix = "blue"
        elif ch == 1:
            suffix = "violet"
        else:
            suffix = f"ch{ch}"

        np.save(os.path.join(processPath, f"U_{suffix}.npy"), U)
        np.save(os.path.join(processPath, f"V_{suffix}.npy"), V)
        np.save(os.path.join(processPath, f"imavg_{suffix}.npy"), im_avg)

    np.save(os.path.join(processPath, f"{experiment}_U.npy"), U_list)
    np.save(os.path.join(processPath, f"{experiment}_V.npy"), V_list)
    np.save(os.path.join(processPath, f"{experiment}_imavg.npy"), im_avg_list)

    return U_list, V_list, im_avg_list



In [ ]:
def preprocess_session(config, session):
    print(f"Processing session: {session}")
    rawPath = config["paths"]["raw"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    print(f"Raw path: {rawPath}")
    processPath = config["paths"]["process"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(processPath, exist_ok=True)
    qc_path = config["paths"]["qc"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(qc_path, exist_ok=True)
    timePath = config["paths"]["time"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(timePath, exist_ok=True)

    items = glob(pjoin(rawPath, '202?????-??????-4*'))
    experiments = list(set([os.path.basename(item)[:15] for item in items]))
    print("Experiments found:", experiments)

    for experiment in experiments:
        # 1. Merge two channels
        merged_file = merge_two_channels(rawPath, processPath, timePath, experiment)

        # 2. Outlier detection and correction
        values_470 = np.loadtxt(pjoin(processPath, f"{experiment}-470-Values.csv"), delimiter=',')
        values_405 = np.loadtxt(pjoin(processPath, f"{experiment}-405-Values.csv"), delimiter=',')
        n_frames = min(len(values_470), len(values_405))
        mean_values = np.stack([values_470[:n_frames], values_405[:n_frames]], axis=1)
        out470, out405 = detect_outlier(mean_values, std_thr=config["preprocess"]["outlier_std"], qc_path=qc_path, plot=True)
        # out470, out405 = detect_outlier(mean_values, std_thr=2, plot=True)
        corrected_file = correct_lum_outlier(merged_file, out470, out405, plot=True, qc_path=qc_path)
        print("Outlier removal done")

        # 3. SVD
        U_list, V_list, im_avg_list = svd_widefield(processPath, experiment,corrected_file , downsample_factor=15, max_components=500)


In [5]:
config_path = "/home/lsh/WF_GoNogo/config/A095_config.yaml"
def preprocess_mice(config_path):
    config = load_config(config_path)
    for session in config["sessions"]:
        preprocess_session(config, session)
    print(f"all done")

In [7]:
preprocess_mice(config_path)

Processing session: retino
Raw path: /home/lsh/Data_attention/Transfer learning/DATA_linshu/A095/retino
Experiments found: ['20250906-185140', '20250906-191014']
Merged 20250906-185140 already exists
Outlier dection plot saved to: /home/lsh/Data_attention/Transfer learning/DATA_linshu/A095/retino/QualityControl/outlier_detection.png
Merged frames: 7644
Corrected .bin file saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/A095/retino/process/20250906-185140-wfield/7644_2_512_512_uint16.bin
Outlier correction plot saved to: /home/lsh/Data_attention/Transfer learning/DATA_linshu/A095/retino/QualityControl/outlier_correction.png
Outlier removal done
Processing SVD channel 1/2...
Processing SVD channel 2/2...


FileNotFoundError: [Errno 2] No such file or directory: '/home/lsh/Data_attention/Transfer learning/DATA_linshu/A095/retino/time/widefield_timestamps_blue.npy'